# EN4554 Assignment 1, Part 2: Tool-calling agent

**Model:** Qwen2.5-0.5B-Instruct. **Tools:** `calculator`, `unit_convert`, `get_weather`, `lookup_order` (the last two use made-up data).

Pipeline: build the agent loop → generate the data → evaluate the baseline → LoRA fine-tune (loss on assistant turns only) → evaluate again → failure analysis.

## 0. Setup

In [ ]:
# !pip install -q "transformers>=4.46" "peft>=0.13" datasets accelerate

In [ ]:
import os, re, json, math, random, copy
from collections import Counter, defaultdict
import numpy as np
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/EN4554_A1/part2"
else:
    OUT_DIR = "./outputs/part2"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
CFG = dict(
    model_id   = "Qwen/Qwen2.5-0.5B-Instruct",
    n_train    = 1200,
    n_test     = 100,
    frac_no_tool = 0.2,
    frac_two_step = 0.1,
    max_steps  = 4,          # agent loop cap
    max_new_tokens = 128,
    max_len    = 768,
    # LoRA
    r = 16, alpha = 32, dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lr = 2e-4, epochs = 2, batch_size = 8, grad_accum = 2,
)

## 1. Tools (made-up data)

In [ ]:
WEATHER_DB = {  # fictional cities -> the model cannot answer from memory
    "Zentoria":  {"temp_c": 17, "condition": "foggy"},
    "Velmora":   {"temp_c": 29, "condition": "sunny"},
    "Ashbrook":  {"temp_c": 4,  "condition": "snow"},
    "Korrin Bay":{"temp_c": 22, "condition": "light rain"},
    "Pellucid":  {"temp_c": 11, "condition": "windy"},
    "Drovane":   {"temp_c": 33, "condition": "humid"},
}
ORDER_DB = {f"ORD-{1000+i}": {"status": random.choice(["shipped", "processing", "delivered", "cancelled"]),
                              "item": random.choice(["headphones", "desk lamp", "backpack", "keyboard", "water bottle"]),
                              "eta_days": random.randint(0, 9)} for i in range(60)}

UNITS = {  # conversion to a base unit per dimension
    "length": {"m": 1, "km": 1000, "cm": 0.01, "mile": 1609.344, "ft": 0.3048, "inch": 0.0254},
    "mass":   {"kg": 1, "g": 0.001, "lb": 0.45359237, "oz": 0.028349523125},
}

def calculator(expression: str) -> str:
    # Evaluates an arithmetic expression. Args: expression: e.g. "(3+4)*2"
    if not re.fullmatch(r"[\d\s\.\+\-\*/\(\)%]+", expression):
        return "error: invalid expression"
    try:
        return str(round(eval(expression, {"__builtins__": {}}), 4))
    except Exception as e:
        return f"error: {e}"

def unit_convert(value: float, from_unit: str, to_unit: str) -> str:
    for dim in UNITS.values():
        if from_unit in dim and to_unit in dim:
            return str(round(float(value) * dim[from_unit] / dim[to_unit], 4))
    if {from_unit, to_unit} <= {"c", "f"}:
        v = float(value)
        return str(round(v * 9/5 + 32 if from_unit == "c" else (v - 32) * 5/9, 2))
    return "error: unsupported units"

def get_weather(city: str) -> str:
    w = WEATHER_DB.get(city.strip().title())
    return json.dumps(w) if w else "error: unknown city"

def lookup_order(order_id: str) -> str:
    o = ORDER_DB.get(order_id.strip().upper())
    return json.dumps(o) if o else "error: order not found"

TOOL_FNS = {f.__name__: f for f in [calculator, unit_convert, get_weather, lookup_order]}

In [ ]:
# JSON schemas passed to the chat template via tools=...
TOOLS = [
    {"type": "function", "function": {"name": "calculator", "description": "Evaluate an arithmetic expression.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "e.g. (3+4)*2"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "unit_convert", "description": "Convert a value between units (m, km, cm, mile, ft, inch, kg, g, lb, oz, c, f).",
        "parameters": {"type": "object", "properties": {"value": {"type": "number"}, "from_unit": {"type": "string"}, "to_unit": {"type": "string"}},
                       "required": ["value", "from_unit", "to_unit"]}}},
    {"type": "function", "function": {"name": "get_weather", "description": "Get current weather for a city.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "lookup_order", "description": "Look up an order by its ID (format ORD-XXXX).",
        "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
]
SYSTEM = "You are a helpful assistant. Use a tool when it is needed; otherwise answer directly."

## 2. Model + agent loop

In [ ]:
tok = AutoTokenizer.from_pretrained(CFG["model_id"])
def load_model():
    return AutoModelForCausalLM.from_pretrained(CFG["model_id"], torch_dtype=torch.float32).to(DEVICE)
model = load_model()

# Look at how the template renders tools and tool calls
print(tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": "hi"}],
                              tools=TOOLS, tokenize=False, add_generation_prompt=True))

In [ ]:
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)

def parse_tool_calls(text):
    # Returns (calls, valid). calls = [{"name":..., "arguments": {...}}]; valid=False if a <tool_call> block is malformed.
    # TODO (critical): find all <tool_call> blocks, json.loads each, check it has "name" (str) and "arguments" (dict).
    #   Count it as invalid (format error) if the JSON fails to parse or the keys are missing.
    raise NotImplementedError

def execute(call):
    fn = TOOL_FNS.get(call["name"])
    if fn is None:
        return f"error: unknown tool {call['name']}"
    try:
        return fn(**call["arguments"])
    except Exception as e:
        return f"error: {e}"

@torch.no_grad()
def generate(model, messages):
    prompt = tok.apply_chat_template(messages, tools=TOOLS, tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(**enc, max_new_tokens=CFG["max_new_tokens"], do_sample=False)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def run_agent(model, user_msg, verbose=False):
    # Goal -> Act -> Observe loop. Returns the full trace for evaluation.
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    trace = dict(calls=[], valid_format=True, final=None, raw=[])
    for _ in range(CFG["max_steps"]):
        out = generate(model, messages)
        trace["raw"].append(out)
        calls, valid = parse_tool_calls(out)
        trace["valid_format"] &= valid
        if not calls:                          # plain-text answer -> done
            trace["final"] = out
            break
        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            result = execute(c)
            trace["calls"].append({**c, "result": result})
            messages.append({"role": "tool", "name": c["name"], "content": result})
        if verbose: print(out, "->", [c["result"] for c in trace["calls"][-len(calls):]])
    return trace

run_agent(model, "What's the weather in Velmora?", verbose=True)

## 3. Data (template-generated)

Each example: `{"user", "calls": [ {name, arguments}, ... ], "answer"}`, with `calls=[]` for no-tool requests.
**Test templates are written separately from the training templates**, so the test phrasings are never seen in training.

In [ ]:
TRAIN_TEMPLATES = {
    "get_weather":  ["What's the weather in {city}?", "How warm is it in {city} right now?", "Is it raining in {city}?"],
    "lookup_order": ["Where is my order {oid}?", "Can you check the status of {oid}?", "Has {oid} shipped yet?"],
    "calculator":   ["What is {a} times {b}?", "Compute {a} + {b} * {c}.", "What's {a} divided by {b}?"],
    "unit_convert": ["Convert {v} {fu} to {tu}.", "How many {tu} is {v} {fu}?"],
    "no_tool":      ["What is the capital of France?", "Tell me a short joke.", "What does CPU stand for?", "Say hello in Spanish."],
    "two_step":     ["What's the temperature in {city} in Fahrenheit?", "How many hours until order {oid} arrives?"],
}
TEST_TEMPLATES = {
    # TODO: write genuinely different phrasings (not paraphrases of the train strings above)
    "get_weather":  ["Any idea what conditions are like over in {city} today?"],
    "lookup_order": ["I placed {oid} last week; what's going on with it?"],
    "calculator":   ["Quick math: {a} multiplied by {b} please."],
    "unit_convert": ["I've got {v} {fu}; what's that in {tu}?"],
    "no_tool":      ["Who wrote Romeo and Juliet?"],
    "two_step":     ["Give me the {city} temperature but in °F."],
}
NO_TOOL_ANSWERS = {}   # TODO: reference answers (or keywords) for the no-tool prompts, used by task success

def sample_example(templates, rng):
    # TODO (critical): pick a category using CFG frac_no_tool / frac_two_step, fill in the slots with rng,
    #   and return {"user": str, "calls": [{"name", "arguments"}], "answer": str} where the gold answer
    #   is computed by actually running the tools (two_step = chain two calls, e.g. get_weather -> unit_convert c->f).
    raise NotImplementedError

def build_split(templates, n, seed):
    rng = random.Random(seed)
    return [sample_example(templates, rng) for _ in range(n)]

train_set = build_split(TRAIN_TEMPLATES, CFG["n_train"], SEED)
test_set  = build_split(TEST_TEMPLATES,  CFG["n_test"],  SEED + 1)
print(len(train_set), len(test_set)); train_set[:3]

In [ ]:
def to_chat(ex):
    # Convert an example to a full chat transcript for SFT (the same format the agent loop uses)
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["user"]}]
    for c in ex["calls"]:   # one call per assistant turn (two-step = two turns)
        msgs.append({"role": "assistant", "content": "", "tool_calls": [{"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}]})
        msgs.append({"role": "tool", "name": c["name"], "content": execute(c)})
    msgs.append({"role": "assistant", "content": ex["answer"]})
    return msgs

print(tok.apply_chat_template(to_chat(train_set[0]), tools=TOOLS, tokenize=False))

## 4. Evaluation

In [ ]:
def score(ex, trace):
    needs_tool = len(ex["calls"]) > 0
    made_call = len(trace["calls"]) > 0
    # TODO (critical): compute the four metrics for one example:
    #   call_valid    : (only if needs_tool) the model emitted >=1 call and all were well-formed
    #   call_accuracy : (only if needs_tool) predicted (name, arguments) match gold (normalise case/whitespace/number types)
    #   abstained     : (only if not needs_tool) no call was made
    #   task_success  : the final answer contains the gold answer (numbers / key fields)
    # Use None for metrics that don't apply, so they are skipped in the averages.
    raise NotImplementedError

def evaluate(model, data, name):
    rows = []
    for ex in tqdm(data, desc=name):
        tr = run_agent(model, ex["user"])
        rows.append({**ex, **score(ex, tr), "trace": tr})
    df = pd.DataFrame(rows)
    summary = {m: df[m].dropna().astype(float).mean() * 100 for m in ["call_valid", "call_accuracy", "abstained", "task_success"]}
    print(name, {k: round(v, 1) for k, v in summary.items()})
    return summary, df

In [ ]:
base_summary, base_df = evaluate(model, test_set, "baseline")
base_df.to_pickle(f"{OUT_DIR}/baseline_eval.pkl")

## 5. LoRA fine-tuning (loss on assistant turns only)

In [ ]:
def tokenize_with_assistant_mask(ex):
    text = tok.apply_chat_template(to_chat(ex), tools=TOOLS, tokenize=False)
    enc = tok(text, truncation=True, max_length=CFG["max_len"], return_offsets_mapping=True)
    labels = [-100] * len(enc["input_ids"])
    # TODO (critical): un-mask (copy input_ids into labels for) only the tokens inside assistant turns,
    #   i.e. each span after "<|im_start|>assistant\n" up to and including the closing "<|im_end|>".
    #   Use re.finditer on `text` + enc["offset_mapping"] to map character spans to token indices.
    #   System, user and tool-result tokens must stay -100.
    raise NotImplementedError
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

train_tok = [tokenize_with_assistant_mask(ex) for ex in train_set]
# sanity check: decoded unmasked labels should only show tool calls and final answers
# print(tok.decode([t for t in train_tok[0]["labels"] if t != -100]))

In [ ]:
def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = lambda x, v: x + [v] * (L - len(x))
    return {
        "input_ids":      torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
        "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch]),
        "labels":         torch.tensor([pad(b["labels"], -100) for b in batch]),
    }

set_seed()
lora_model = get_peft_model(load_model(), LoraConfig(
    r=CFG["r"], lora_alpha=CFG["alpha"], lora_dropout=CFG["dropout"],
    target_modules=CFG["target_modules"], task_type="CAUSAL_LM"))
lora_model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=f"{OUT_DIR}/lora", num_train_epochs=CFG["epochs"],
    per_device_train_batch_size=CFG["batch_size"], gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["lr"], lr_scheduler_type="cosine", warmup_ratio=0.03,
    fp16=True, logging_steps=10, save_strategy="epoch", save_total_limit=1,
    report_to="none", seed=SEED,
)
trainer = Trainer(model=lora_model, args=args, train_dataset=train_tok, data_collator=collate)
trainer.train()
lora_model.save_pretrained(f"{OUT_DIR}/lora/adapter")

In [ ]:
lora_summary, lora_df = evaluate(lora_model, test_set, "lora")
lora_df.to_pickle(f"{OUT_DIR}/lora_eval.pkl")

results = pd.DataFrame({"Baseline": base_summary, "LoRA": lora_summary}).round(1)
results.to_csv(f"{OUT_DIR}/results.csv"); results

## 6. Failure analysis (≥10 failures per model)

In [ ]:
CATEGORIES = ["wrong_format", "wrong_tool", "wrong_args", "unnecessary_call", "missed_call", "ignored_result", "ok"]

def categorize(row):
    # TODO (critical): assign one category per example using the gold calls and the trace, e.g.
    #   invalid format -> wrong_format; no-tool request but called -> unnecessary_call; tool needed but none called -> missed_call;
    #   wrong name -> wrong_tool; right name but wrong args -> wrong_args; correct call but final answer wrong -> ignored_result.
    raise NotImplementedError

for df in (base_df, lora_df):
    df["category"] = df.apply(categorize, axis=1)

cmp = pd.DataFrame({"Baseline": base_df["category"].value_counts(), "LoRA": lora_df["category"].value_counts()}).fillna(0).astype(int)
cmp

In [ ]:
# Fixed / not fixed / introduced by LoRA (same test example, same index)
trans = pd.crosstab(base_df["category"], lora_df["category"], rownames=["baseline"], colnames=["lora"])
fixed      = ((base_df["category"] != "ok") & (lora_df["category"] == "ok")).sum()
not_fixed  = ((base_df["category"] != "ok") & (lora_df["category"] != "ok")).sum()
introduced = ((base_df["category"] == "ok") & (lora_df["category"] != "ok")).sum()
print(f"fixed={fixed}  not fixed={not_fixed}  introduced={introduced}")
trans

In [ ]:
# Print >=10 failures per model for the write-up
def show_failures(df, name, n=10):
    print(f"===== {name} failures =====")
    for _, r in df[df["category"] != "ok"].head(n).iterrows():
        print(f"[{r['category']}] {r['user']}\n  gold: {r['calls']} -> {r['answer']}\n  model: {r['trace']['raw']}\n")

show_failures(base_df, "baseline"); show_failures(lora_df, "lora")

## 7. Notes for the write-up

- Setup: model, tools, data sizes (train/test, % no-tool, % two-step), LoRA settings, seed 42.
- Baseline vs LoRA table (above).
- Failure categories: which LoRA fixed, which remain, which it introduced.
- 2–3 sentences: would a better prompt alone (few-shot examples of the `<tool_call>` format) have been enough?